#### Imports

In [ ]:
# import util functions from external util file (in the same folder)
# also imports all necessary libraries
%run util.py

# run for different font sizes
plt.rcParams.update({'font.size': 10})  # Or whatever size you like

# Set to True if you want to open excel files through xlwings in the background
# Note: This starts excel. Excel is required on your machine
if False:
    app = xw.App(visible=True)
else:
    app = xw.App()

# SET UP

This setup will copy the data set (physical copy of the excel). The subsequent code will then only manipulate this copy.

Notes:

- The data file MUST NOT contain empty rows or empty columns. Especially the first column and row must be filled. Otherwise xlwings has trouble detecting the full data range

In [ ]:
# xlwings setup

# copy the original file to a new one with the appropriate datetime stamp, so the original file is never written to
ts = time.time()
timestamp = datetime.datetime.fromtimestamp(ts).strftime('%Y%m%d_%H%M%S')
original_path = f'./methodology_data.xlsx'
copied_path =   f'./methodology_data_MC_{timestamp}.xlsx'
shutil.copyfile(original_path, copied_path)

 # Name of the sheet containing data
sheet_name = 'data' 

num_header_rows = 1 # number of rows with metadata or headers
num_header_columns = 3 # number of columns with metadata or headers

# Connect to Excel
# THIS OPENS AND EDITS THE EXCEL FILE ON YOUR COMPUTER!
wb = app.books.open(copied_path)
sheet = wb.sheets[sheet_name]

In [ ]:
def update_range(sheet):
    # Determine the range of columns and results
    data_range = sheet.range("A1").expand()  # Automatically detect data range
    global num_columns 
    num_columns = data_range.columns.count  # Total number of columns
    global num_rows
    num_rows = data_range.rows.count  # Total number of columns

num_columns = 0
num_rows = 0
update_range(sheet)

print(num_columns)
print(num_rows)

### Data Preparation

### Store the original ER

In [ ]:
# copy the original numbers of the variable ER into a new variable ER_original underneath (if it doesn't exist already)
row_ER = get_row_by_variable("ER", sheet, num_rows)

if (get_row_by_variable("ER_original", sheet, num_rows) == -1):
    row_ER_original = insert_row_after(num_rows, "ER_original", sheet, num_rows)
    update_range(sheet)
    sheet.range((row_ER_original, num_header_columns + 1)).value = sheet.range((row_ER, num_header_columns + 1), (row_ER, num_columns)).value

### Update variables

In [ ]:
# The GWP of CH4 has been updated in recent years. 
# Since we're investigating the difference between what old projects estimated and what they should have estimated, 
# we have to update the GWP value with the current default value 28 for all projects in order to be able to compare them.
# This is a source of undercrediting

row_GWP_CH4 = get_row_by_variable("GWP_CH4", sheet, num_rows)
sheet.range((row_GWP_CH4, num_header_columns + 1)).value = [28] * (num_columns - num_header_columns)

# Some projects use an uncertainty discount for inherent uncertainty. (usually between 0.9 and 0.95) Since we are estimating uncertainty 
# we have to set this factor to 1 so we don't double count uncertainty. 

row_UF_b = get_row_by_variable("UF_b", sheet, num_rows)
sheet.range((row_UF_b, num_header_columns + 1)).value = [1] * (num_columns - num_header_columns)
row_UF_b_coal = get_row_by_variable("UF_b_coal", sheet, num_rows)
sheet.range((row_UF_b_coal, num_header_columns + 1)).value = [1] * (num_columns - num_header_columns)
row_UF_p_coal = get_row_by_variable("UF_p_coal", sheet, num_rows)
sheet.range((row_UF_p_coal, num_header_columns + 1)).value = [1] * (num_columns - num_header_columns)


# Monte Carlo Simulation

### f_NRB

This cell handles the excel calculations. 

New rows will be added to the end of the data set.

In [ ]:
# MC simulation for fnrb 

fnrb_lower = 0.191
fnrb_upper = 0.396
frnb_mode = 0.296

num_trials = 1000

# store original values of fnrb
if (get_row_by_variable("f_NRB_original", sheet, num_rows) == -1): #add row if doesn't exist yet
    row_fnrb = get_row_by_variable("f_NRB", sheet, num_rows)
    row_original_fnrb = insert_row_after(num_rows, "f_NRB_original", sheet, num_rows)
    update_range(sheet)
    row_fnrb = get_row_by_variable("f_NRB", sheet, num_rows)
    row_original_fnrb = get_row_by_variable("f_NRB_original", sheet, num_rows)
    sheet.range((row_original_fnrb, num_header_columns + 1)).value = sheet.range((row_fnrb, num_header_columns + 1), (row_fnrb, num_columns)).value

row_fnrb = get_row_by_variable("f_NRB", sheet, num_rows)
row_original_fnrb = get_row_by_variable("f_NRB_original", sheet, num_rows)

# results (new rows that will be added  automatically underneath ER_oroginal (only if they don't exist yet)
row_ER_original = get_row_by_variable("ER_original", sheet, num_rows)
if (get_row_by_variable("ER_mc_fnrb", sheet, num_rows) == -1):
    row_ER_mc_fnrb = insert_row_after(num_rows, "ER_mc_fnrb", sheet, num_rows)
    update_range(sheet)
if (get_row_by_variable("ER_mc_fnrb_factor", sheet, num_rows) == -1):
    row_ER_mc_fnrb_factor = insert_row_after(num_rows, "ER_mc_fnrb_factor", sheet, num_rows)
    update_range(sheet)
if (get_row_by_variable("ER_mc_fnrb_factor_variance", sheet, num_rows) == -1):
    row_ER_mc_fnrb_factor_variance = insert_row_after(num_rows, "ER_mc_fnrb_factor_variance", sheet, num_rows)
    update_range(sheet)

# update number of rows
update_range(sheet)

# update relevant row numbers
row_ER = get_row_by_variable("ER", sheet, num_rows)
row_ER_original = get_row_by_variable("ER_original", sheet, num_rows)
row_fnrb = get_row_by_variable("f_NRB", sheet, num_rows)
row_original_fnrb = get_row_by_variable("f_NRB_original", sheet, num_rows)
row_ER_mc_fnrb = get_row_by_variable("ER_mc_fnrb", sheet, num_rows)
row_ER_mc_fnrb_factor = get_row_by_variable("ER_mc_fnrb_factor", sheet, num_rows)
row_ER_mc_fnrb_factor_variance = get_row_by_variable("ER_mc_fnrb_factor_variance", sheet, num_rows)

# prepare empty lists for MC results
# contains one array per iteration
# in each array, coordinates correspond to MC ERs for the project at this coordinate
monte_carlo_results = []
monte_carlo_results_factor = []

ER_original = np.asarray(sheet.range((row_ER_original, num_header_columns + 1), (row_ER_original, num_columns)).value)


# Monte Carlo Simulation
for trial in range(num_trials):
    sheet.range((row_fnrb, num_header_columns + 1)).value = sample_triangular_distribution(fnrb_lower, fnrb_upper, frnb_mode, num_columns - num_header_columns)

    # copy the calculated results and append them to the MC list
    result = sheet.range((row_ER, num_header_columns + 1), (row_ER, num_columns)).value
    monte_carlo_results.append(result)
    monte_carlo_results_factor.append(ER_original / np.asarray(result))

# average between trials
ER_results = np.sum(monte_carlo_results, axis=0) / num_trials
ER_variance = np.var(monte_carlo_results_factor, axis=0)
ER_original = sheet.range((row_ER_original, num_header_columns + 1), (row_ER_original, num_columns)).value
ER_factor = np.asarray(ER_original) / ER_results

sheet.range((row_ER_mc_fnrb, num_header_columns + 1)).value = ER_results
sheet.range((row_ER_mc_fnrb_factor, num_header_columns + 1)).value = ER_factor
sheet.range((row_ER_mc_fnrb_factor_variance, num_header_columns + 1)).value = ER_variance


# restore fnrb row to original values (Also restores ER to original value)
sheet.range((row_fnrb, num_header_columns + 1)).value = sheet.range((row_original_fnrb, num_header_columns + 1), (row_original_fnrb, num_columns)).value



This cell extracts the appropriate value from the excel and calculates mean and variance for each methodology.
Then, it creates a plot for it.

In [ ]:
mean_and_variance_fnrb = get_mean_and_variance("ER_mc_fnrb_factor", "ER_mc_fnrb_factor_variance", sheet, num_rows, num_columns, num_header_columns)

plot_methodologies(mean_and_variance_fnrb, "MC Simulation Results for f_NRB")

### Functionality

Also called share of active households or devices in operation, HH_active in the data set. 

Approach: We set the HH_active to 1. Some projects used survey values to estimate HH_active but arrived at extremely high values (99%) (which results in the assumption that 99% of digesters work optimally all year, which would be unteresting for usage.). Since we want to model this with values supported by literature we need to set HH_active to 1 to compare the issued credits with updated credits. 


In [ ]:
row_HH_active = get_row_by_variable("HH_active", sheet, num_rows)
sheet.range((row_HH_active, num_header_columns + 1)).value = [1] * (num_columns - num_header_columns)

# MC simulation for functionality 
# Observations from literature review
functionality_discrete_dist = np.array([0.72,0.60,0.40,0.60,0.40,0.80,0.40,0.80,0.82,0.71,0.63,0.66,0.60,0.40,0.60,0.86,0.95,0.60,0.90,0.10,0.37,0.69,0.55,0.614,0.69,0.85,0.91,0.92,0.40])

functionality_lower = 0.1
functionality_upper = 0.9
functionality_mode = 0.6

num_trials = 1000

# results (new rows that will be added  automatically underneath ER_oroginal
row_ER_original = get_row_by_variable("ER_original", sheet, num_rows)
if (get_row_by_variable("ER_mc_functionality", sheet, num_rows) == -1):
    row_ER_mc_functionality = insert_row_after(num_rows, "ER_mc_functionality", sheet, num_rows)
    update_range(sheet)
if (get_row_by_variable("ER_mc_functionality_factor", sheet, num_rows) == -1):
    row_ER_mc_functionality_factor = insert_row_after(num_rows, "ER_mc_functionality_factor", sheet, num_rows)
    update_range(sheet)
if (get_row_by_variable("ER_mc_functionality_factor_variance", sheet, num_rows) == -1):
    row_ER_mc_functionality_variance = insert_row_after(num_rows, "ER_mc_functionality_factor_variance", sheet, num_rows)
    update_range(sheet)

# update number of rows
update_range(sheet)

# update relevant row numbers
row_ER = get_row_by_variable("ER", sheet, num_rows)
row_ER_original = get_row_by_variable("ER_original", sheet, num_rows)
row_ER_mc_functionality = get_row_by_variable("ER_mc_functionality", sheet, num_rows)
row_ER_mc_functionality_factor = get_row_by_variable("ER_mc_functionality_factor", sheet, num_rows)
row_ER_mc_functionality_factor_variance = get_row_by_variable("ER_mc_functionality_factor_variance", sheet, num_rows)

# prepare empty lists for MC results
# contains one array per iteration
# in each array, coordinates correspond to MC ERs for the project at this coordinate
monte_carlo_results = []
monte_carlo_results_factor = []

ER_original = np.asarray(sheet.range((row_ER_original, num_header_columns + 1), (row_ER_original, num_columns)).value)

# Monte Carlo Simulation
for trial in range(num_trials):
    # Replace all values in this row with MC samples (excluding the header columns)
    #functionality_sample = sample_triangular_distribution(functionality_lower, functionality_upper, functionality_mode, num_columns - num_header_columns)
    functionality_sample = sample_from_discrete_distribution(functionality_discrete_dist, num_columns - num_header_columns)
    ER_values = np.asarray(sheet.range((row_ER, num_header_columns + 1), (row_ER, num_columns)).value)
    
    # copy the calculated results and append them to the MC list
    monte_carlo_results.append(functionality_sample * ER_values)
    monte_carlo_results_factor.append(ER_original / (functionality_sample * ER_values))

# average between trials
ER_results = np.sum(monte_carlo_results, axis=0) / num_trials
ER_variance = np.var(monte_carlo_results_factor, axis=0)
ER = sheet.range((row_ER, num_header_columns + 1), (row_ER, num_columns)).value
ER_original = sheet.range((row_ER_original, num_header_columns + 1), (row_ER_original, num_columns)).value
ER_factor = np.asarray(ER_original) / ER_results

sheet.range((row_ER_mc_functionality, num_header_columns + 1)).value = ER_results
sheet.range((row_ER_mc_functionality_factor, num_header_columns + 1)).value = ER_factor
sheet.range((row_ER_mc_functionality_factor_variance, num_header_columns + 1)).value = ER_variance

In [ ]:
mean_and_variance_functionality = get_mean_and_variance("ER_mc_functionality_factor","ER_mc_functionality_factor_variance", sheet, num_rows, num_columns, num_header_columns)

plot_methodologies(mean_and_variance_functionality, "MC Simulation Results for Functionality")

### Combined

In [ ]:
row_HH_active = get_row_by_variable("HH_active", sheet, num_rows)
sheet.range((row_HH_active, num_header_columns + 1)).value = [1] * (num_columns - num_header_columns)

# MC simulation for combined

functionality_lower = 0.1
functionality_upper = 0.9
functionality_mode = 0.6

fnrb_lower = 0.191
fnrb_upper = 0.396
frnb_mode = 0.296

num_trials = 1000

# store original values of fnrb
if (get_row_by_variable("f_NRB_original", sheet, num_rows) == -1): #add row if doesn't exist yet
    row_fnrb = get_row_by_variable("f_NRB", sheet, num_rows)
    row_original_fnrb = insert_row_after(num_rows, "f_NRB_original", sheet, num_rows)
    update_range(sheet)
    row_fnrb = get_row_by_variable("f_NRB", sheet, num_rows)
    row_original_fnrb = get_row_by_variable("f_NRB_original", sheet, num_rows)
    sheet.range((row_original_fnrb, num_header_columns + 1)).value = sheet.range((row_fnrb, num_header_columns + 1), (row_fnrb, num_columns)).value



# results (new rows that will be added  automatically underneath ER_oroginal
row_ER_original = get_row_by_variable("ER_original", sheet, num_rows)
if (get_row_by_variable("ER_mc_combined", sheet, num_rows) == -1):
    row_ER_mc_combined = insert_row_after(num_rows, "ER_mc_combined", sheet, num_rows)
    update_range(sheet)
if (get_row_by_variable("ER_mc_combined_factor", sheet, num_rows) == -1):
    row_ER_mc_combined_factor = insert_row_after(num_rows, "ER_mc_combined_factor", sheet, num_rows)
    update_range(sheet)
if (get_row_by_variable("ER_mc_combined_factor_variance", sheet, num_rows) == -1):
    row_ER_mc_combined_factor_variance = insert_row_after(num_rows, "ER_mc_combined_factor_variance", sheet, num_rows)
    update_range(sheet)

# update number of rows
update_range(sheet)

# update relevant row numbers
row_ER = get_row_by_variable("ER", sheet, num_rows)
row_ER_original = get_row_by_variable("ER_original", sheet, num_rows)
row_ER_mc_combined_factor = get_row_by_variable("ER_mc_combined_factor", sheet, num_rows)
row_ER_mc_combined = get_row_by_variable("ER_mc_combined", sheet, num_rows)
row_ER_mc_combined_factor_variance = get_row_by_variable("ER_mc_combined_factor_variance", sheet, num_rows)
row_fnrb = get_row_by_variable("f_NRB", sheet, num_rows)
row_original_fnrb = get_row_by_variable("f_NRB_original", sheet, num_rows)

# prepare empty lists for MC results
# contains one array per iteration
# in each array, coordinates correspond to MC ERs for the project at this coordinate
monte_carlo_results = []
monte_carlo_results_factor = []

ER_original = np.asarray(sheet.range((row_ER_original, num_header_columns + 1), (row_ER_original, num_columns)).value)

# Monte Carlo Simulation
for trial in range(num_trials):
    # fnrb
    sheet.range((row_fnrb, num_header_columns + 1)).value = sample_triangular_distribution(fnrb_lower, fnrb_upper, frnb_mode, num_columns - num_header_columns)

    # functionality
    functionality_sample = sample_triangular_distribution(functionality_lower, functionality_upper, functionality_mode, num_columns - num_header_columns)
    ER_values = np.asarray(sheet.range((row_ER, num_header_columns + 1), (row_ER, num_columns)).value)
    
    # copy the calculated results and append them to the MC list
    monte_carlo_results.append(functionality_sample * ER_values)
    
    result = sheet.range((row_ER, num_header_columns + 1), (row_ER, num_columns)).value
    
    monte_carlo_results_factor.append(ER_original / (functionality_sample * np.asarray(result)))

# average between trials
ER_results = np.sum(monte_carlo_results, axis=0) / num_trials
ER_variance = np.var(monte_carlo_results_factor, axis=0)
ER_original = sheet.range((row_ER_original, num_header_columns + 1), (row_ER_original, num_columns)).value
ER_factor = np.asarray(ER_original) / ER_results

sheet.range((row_ER_mc_combined, num_header_columns + 1)).value = ER_results
sheet.range((row_ER_mc_combined_factor, num_header_columns + 1)).value = ER_factor
sheet.range((row_ER_mc_combined_factor_variance, num_header_columns + 1)).value = ER_variance

# restore fnrb row to original values (Also restores ER to original value)
sheet.range((row_fnrb, num_header_columns + 1)).value = sheet.range((row_original_fnrb, num_header_columns + 1), (row_original_fnrb, num_columns)).value


In [ ]:
mean_and_variance_functionality = get_mean_and_variance("ER_mc_combined_factor","ER_mc_combined_factor_variance", sheet, num_rows, num_columns, num_header_columns)

plot_methodologies(mean_and_variance_functionality, "MC Simulation Combined")